# RAG Concepts: Embeddings, Retrieval & Chunking

Retrieval-Augmented Generation (RAG) is the dominant architecture for grounding LLMs in private, current, or domain-specific knowledge — exactly the situation in financial services, where analysts work with proprietary filings, earnings data, and regulatory documents that postdate any model's training cutoff. Rather than asking the model to recall facts it may never have seen, we retrieve the relevant passages at query time and hand them to the model as context.

The notebook is structured around three interlocking ideas: (1) **why RAG** — what problems it solves and why fine-tuning alone is insufficient; (2) **dense embeddings** — how text is mapped to vectors such that semantic similarity becomes geometric proximity; (3) **chunking strategies** — how we split long documents to maximize retrieval precision. Understanding all three is necessary to reason about where a RAG pipeline can fail and how to fix it.

By the end we'll have a working minimal RAG pipeline over a sample financial corpus using only numpy — no vector database yet. We replace numpy with ChromaDB and real SEC filings in [notebook 07](/courses/llm-eng/07-rag-pipeline.html).

Setup:

In [ ]:
#| echo: false
import os, json
import numpy as np
from dotenv import load_dotenv
load_dotenv()

import openai
from pydantic import BaseModel
from typing import Optional, Type

PRICES = {
    "gpt-4o":      {"input": 2.50,  "output": 10.00},
    "gpt-4o-mini": {"input": 0.15,  "output": 0.60},
}

class LLMClient:
    def __init__(self, model="gpt-4o-mini", temperature=0.0):
        self.model = model; self.temperature = temperature
        self._client = openai.OpenAI()
        self._in = 0; self._out = 0

    def complete(self, messages, *, response_format=None):
        if response_format is not None:
            resp = self._client.beta.chat.completions.parse(
                model=self.model, messages=messages,
                temperature=self.temperature, response_format=response_format)
        else:
            resp = self._client.chat.completions.create(
                model=self.model, messages=messages, temperature=self.temperature)
        if resp.usage:
            self._in += resp.usage.prompt_tokens; self._out += resp.usage.completion_tokens
        if response_format is not None: return resp.choices[0].message.parsed
        return resp.choices[0].message.content

    @property
    def total_cost(self):
        if self.model not in PRICES: return 0.0
        p = PRICES[self.model]
        return (self._in * p["input"] + self._out * p["output"]) / 1_000_000

llm = LLMClient()

## Why RAG?

**Parametric vs. non-parametric memory.** An LLM's weights encode the statistical patterns of its training corpus — this is **parametric memory**. Once training ends, that knowledge is frozen. RAG adds **non-parametric memory**: a retrieval system that fetches relevant documents at inference time and injects them into the context window. The model's job shifts from "recall this fact" to "read and synthesize these passages."

<br>

**Three problems RAG solves.** (1) **Knowledge cutoff** — earnings reports, regulatory updates, and proprietary research simply do not exist in pretrained weights. A model trained through early 2024 cannot answer questions about a Q3 2024 10-K filing no matter how well it was trained. (2) **Hallucination** — when the model lacks knowledge it tends to confabulate plausible-sounding facts. With retrieved context and a well-written system prompt, the model can be instructed to cite sources and refuse to speculate beyond them. (3) **Auditability** — compliance teams at financial institutions need to trace every claim to a source document. With RAG, every answer can be accompanied by the exact passage that supports it; with a fully parametric model there is nothing to point to.

<br>

**Why not just fine-tune?** Fine-tuning embeds knowledge into weights — it is expensive (full training runs or LoRA adapters still cost GPU hours), cannot update in real time, and cannot cite sources. A RAG index can be updated by appending new documents in minutes; a fine-tuned model requires a new training run. For knowledge-heavy, compliance-sensitive applications in financial services, RAG is almost always the right first choice. Fine-tuning is complementary: it teaches the model *style and format*, while RAG provides the *facts*.

<br>

**The two-phase pipeline.** RAG operates in two phases. **Indexing** happens offline: load documents → split into chunks → embed each chunk → store vectors in a vector database. **Querying** happens online at inference time: embed the user query → retrieve the top-$k$ similar chunks → construct a context prompt → generate a grounded answer. The latency-sensitive path is querying; indexing can run as a batch job whenever the document corpus changes.

## Dense Embeddings

An embedding maps a piece of text to a fixed-length vector $\mathbf{e} \in \mathbb{R}^d$ such that semantically similar texts have vectors with high **cosine similarity**:

$$\text{sim}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\| \|\mathbf{b}\|}$$

The range is $[-1, 1]$, where $1$ means identical direction. The key insight is that "similar meaning" becomes "similar direction" in this high-dimensional space, enabling geometric search over a potentially large corpus. OpenAI's `text-embedding-3-small` produces 1536-dimensional vectors and is the default we use throughout this series.

The embeddings are produced by an encoder model (typically a transformer) trained on a large corpus with a contrastive objective: passages that co-occur or that human annotators label as semantically related are pulled toward each other; unrelated passages are pushed apart. The result is a space where financial concepts cluster with other financial concepts, legal language clusters with legal language, and so on.

We define a thin `embed` wrapper around the OpenAI embeddings API and a `cosine_sim` helper:

In [ ]:
def embed(texts: list[str], model: str = "text-embedding-3-small") -> np.ndarray:
    """Return embedding matrix of shape (len(texts), d)."""
    client = openai.OpenAI()
    resp = client.embeddings.create(input=texts, model=model)
    vectors = [item.embedding for item in resp.data]
    return np.array(vectors, dtype=np.float32)


def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    """Cosine similarity between two 1-D vectors."""
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-9))

We visualize the cosine similarity matrix over a set of financial phrases to see the learned semantic structure:

In [ ]:
#| code-fold: true
import matplotlib.pyplot as plt
import seaborn as sns

phrases = [
    "interest rate",
    "federal funds rate",
    "bond yield",
    "treasury note",
    "equity return",
    "stock price",
    "merger",
    "acquisition",
]
vecs = embed(phrases)

n = len(phrases)
sim_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(n):
        sim_matrix[i, j] = cosine_sim(vecs[i], vecs[j])

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(
    sim_matrix,
    xticklabels=phrases,
    yticklabels=phrases,
    cmap="Blues",
    vmin=0.3, vmax=1.0,
    annot=True, fmt=".2f",
    ax=ax,
)
ax.set_title("Cosine similarity matrix — financial phrases")
plt.tight_layout()
plt.show()

The heatmap reveals two clusters: rate/yield concepts (top-left block) and corporate action concepts (bottom-right block). This is semantic search working at the vocabulary level — "federal funds rate" and "bond yield" are near-neighbours even though they share no words in common.

We verify that the ordering of similarities is sensible by querying from a fixed anchor phrase:

In [ ]:
test_pairs = [
    ("interest rate", "federal funds rate"),
    ("interest rate", "stock price"),
    ("interest rate", "merger"),
]
v = {p: embed([p])[0] for p in set(w for pair in test_pairs for w in pair)}
for a, b in test_pairs:
    print(f"sim({a!r}, {b!r}) = {cosine_sim(v[a], v[b]):.4f}")

Similarity decreases as semantic distance increases — rate concepts score highest, followed by equity concepts, then M&A. This ordering is what makes semantic search useful for financial Q&A: a question about interest rates retrieves rate-related passages, not merger announcements.

## Chunking Strategies

We can't embed an entire 100-page 10-K into one vector — the embedding would average over too many topics, washing out the signal for any specific query. A question about CET1 capital ratios should retrieve the capital adequacy paragraph, not an undifferentiated summary of the whole filing. But chunks too small lose the surrounding context that makes a passage interpretable. Chunking is one of the highest-leverage design decisions in a RAG system.

There are three common strategies. **Fixed-size chunking** splits by word or character count with an overlap window to avoid cutting sentences at boundaries. It is simple and predictable. **Semantic/paragraph chunking** splits on natural boundaries — double newlines, section headers, sentence terminators — preserving the coherence of each chunk. **Hierarchical chunking** maintains both a small chunk (for precise retrieval) and a larger parent chunk (for full context at generation time); the small chunk is retrieved, but the parent chunk is sent to the LLM. For structured documents like 10-K filings, splitting on section headers ("Item 1A. Risk Factors", "Item 7. MD&A") is often the dominant signal.

We define a sample corpus of 10 short excerpts representing different sections of an SEC 10-K filing:

In [ ]:
# Sample excerpts representing different sections of an SEC 10-K filing
CORPUS = [
    # Risk factors
    "Interest rate risk represents one of our most significant market risks. "
    "A 100 basis point increase in interest rates would reduce the fair value "
    "of our fixed-rate debt portfolio by approximately $2.3 billion.",

    "Credit risk arises from the potential that a counterparty will fail to "
    "perform its obligations. We manage credit risk through diversification, "
    "collateral requirements, and credit limits by counterparty.",

    "Operational risk includes the risk of loss resulting from inadequate or "
    "failed internal processes, people, systems, or external events, including "
    "cybersecurity threats and technology failures.",

    # MD&A
    "Net revenues for the fiscal year were $47.4 billion, an increase of 8% "
    "compared to the prior year. The increase was driven primarily by higher "
    "net interest income reflecting the rising interest rate environment.",

    "Investment banking revenues decreased 23% to $6.1 billion, reflecting "
    "lower advisory fees amid reduced M&A activity and a challenging "
    "environment for equity and debt underwriting.",

    "Return on equity for the year was 12.4%, compared to 15.1% in the prior "
    "year. Book value per share increased to $312.50, up from $290.20.",

    # Capital and liquidity
    "Our Common Equity Tier 1 (CET1) capital ratio was 14.8% at year-end, "
    "well above the regulatory minimum of 4.5% and our internal target of 13%.",

    "We maintain a liquidity coverage ratio (LCR) of 128%, exceeding the "
    "regulatory requirement of 100%. Our high-quality liquid assets totaled "
    "$280 billion at year-end.",

    # Forward guidance
    "Looking ahead to fiscal 2025, management expects continued revenue growth "
    "in the range of 4-6%, supported by a stable rate environment and "
    "improving capital markets activity.",

    "We plan to return $8 billion to shareholders through dividends and share "
    "repurchases in fiscal 2025, subject to regulatory approval and market conditions.",
]

**Fixed-size chunking** splits each text into windows of `chunk_size` words, advancing by `chunk_size - overlap` words at each step so consecutive chunks share a small overlap window:

In [ ]:
def chunk_fixed(
    texts: list[str],
    chunk_size: int = 150,
    overlap: int = 30,
) -> list[dict]:
    """Split texts by word count with overlap. Returns chunks with metadata."""
    chunks = []
    for doc_id, text in enumerate(texts):
        words = text.split()
        start = 0
        while start < len(words):
            end = min(start + chunk_size, len(words))
            chunk_text = " ".join(words[start:end])
            chunks.append({
                "text": chunk_text,
                "doc_id": doc_id,
                "start_word": start,
            })
            if end == len(words):
                break
            start += chunk_size - overlap  # <1>
    return chunks


fixed_chunks = chunk_fixed(CORPUS, chunk_size=40, overlap=8)
print(f"Fixed chunking: {len(fixed_chunks)} chunks")
for i, c in enumerate(fixed_chunks[:3]):
    print(f"  [{i}] doc={c['doc_id']} | {c['text'][:80]}...")

1. Advancing by `chunk_size - overlap` ensures the next chunk begins `overlap` words before the end of the current one — preserving cross-boundary context.

**Paragraph chunking** splits on natural document boundaries — double newlines or, for our single-sentence corpus, treats each entry as its own paragraph:

In [ ]:
def chunk_by_paragraph(texts: list[str]) -> list[dict]:
    """Split on double newlines or treat each text as its own chunk."""
    chunks = []
    for doc_id, text in enumerate(texts):
        paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
        if not paragraphs:
            paragraphs = [text]
        for para in paragraphs:
            chunks.append({"text": para, "doc_id": doc_id})
    return chunks


para_chunks = chunk_by_paragraph(CORPUS)
print(f"Paragraph chunking: {len(para_chunks)} chunks")
print(f"Average chunk length: {sum(len(c['text'].split()) for c in para_chunks)/len(para_chunks):.0f} words")

For our 10-sentence corpus the paragraph strategy produces one chunk per sentence — which is ideal here. For real 10-K filings with structured sections (Item 1A Risk Factors, Item 7 MD&A), document-structure-aware splitting on section headers is the right choice: section boundaries are stronger signals than double newlines and preserve the regulatory context that each section carries.

:::{.callout-note}
Chunk size and strategy are empirically tuned. The right test is to measure **context recall** — does the top-$k$ retrieved set contain the chunk needed to answer the question? We build this measurement in [notebook 04](/courses/llm-eng/04-eval-concepts.html).

:::

## Vector Store and Retrieval

A **vector store** indexes embeddings for fast approximate nearest-neighbor (ANN) search. Production systems use specialized databases — ChromaDB, Pinecone, pgvector — that implement ANN algorithms such as HNSW or IVF-PQ to search millions of vectors in milliseconds. We build a minimal version backed by numpy to demystify what those systems do under the hood before we use them in [notebook 07](/courses/llm-eng/07-rag-pipeline.html).

The key operation is the **batched cosine similarity**: given a query vector $\mathbf{q} \in \mathbb{R}^d$ and a matrix of stored embeddings $\mathbf{M} \in \mathbb{R}^{n \times d}$, we compute all $n$ similarities in one matrix-vector product after row-normalizing $\mathbf{M}$ and normalizing $\mathbf{q}$:

$$\text{scores} = \hat{\mathbf{M}} \, \hat{\mathbf{q}} \in \mathbb{R}^n$$

where $\hat{\mathbf{M}}_{i,:} = \mathbf{M}_{i,:} / \|\mathbf{M}_{i,:}\|$ and $\hat{\mathbf{q}} = \mathbf{q} / \|\mathbf{q}\|.$ This is algorithmically equivalent to computing $n$ individual cosine similarities but executes as a single BLAS call.

We implement `VectorStore` as a simple class with `add` and `retrieve` methods:

In [ ]:
class VectorStore:
    """Minimal in-memory vector store backed by numpy."""

    def __init__(self):
        self.vectors: list[np.ndarray] = []
        self.metadata: list[dict] = []

    def add(self, texts: list[str], metadata: list[dict] | None = None) -> None:
        """Embed and store texts with optional metadata."""
        vecs = embed(texts)
        self.vectors.extend(vecs)
        for i, text in enumerate(texts):
            meta = (metadata[i] if metadata else {})
            self.metadata.append({"text": text, **meta})

    def retrieve(self, query: str, k: int = 3) -> list[dict]:
        """Return top-k chunks by cosine similarity to the query."""
        q_vec = embed([query])[0]                          # <1>
        matrix = np.stack(self.vectors)                   # <2>
        norms = np.linalg.norm(matrix, axis=1, keepdims=True) + 1e-9
        q_norm = np.linalg.norm(q_vec) + 1e-9
        scores = (matrix / norms) @ (q_vec / q_norm)      # <3>
        top_k = np.argsort(scores)[::-1][:k]
        return [
            {"score": float(scores[i]), **self.metadata[i]}
            for i in top_k
        ]

1. We embed the query using the same model as the corpus — the embedding space must be consistent for distances to be meaningful.
2. Stack all stored vectors into a matrix of shape `(n_chunks, d)` for batch cosine computation.
3. Normalise each row and the query, then compute dot products in one matrix-vector multiply — equivalent to $n$ individual cosine similarity computations.

We index the paragraph chunks and run three representative financial queries:

In [ ]:
store = VectorStore()
store.add(
    texts=[c["text"] for c in para_chunks],
    metadata=[{"doc_id": c["doc_id"]} for c in para_chunks],
)
print(f"Indexed {len(store.vectors)} chunks\n")

queries = [
    "What is the CET1 capital ratio?",
    "How did investment banking revenues change?",
    "What are the main operational risks?",
]
for q in queries:
    print(f"Q: {q}")
    results = store.retrieve(q, k=2)
    for r in results:
        print(f"  [{r['score']:.3f}] {r['text'][:90]}...")
    print()

We measure Recall@$k$ — the fraction of queries for which the correct chunk appears in the top-$k$ results — to quantify how retrieval quality degrades as $k$ decreases:

In [ ]:
#| code-fold: true
# Ground-truth: which doc_id contains the answer?
ground_truth = {
    "What is the CET1 capital ratio?": 6,
    "How did investment banking revenues change?": 4,
    "What are the main operational risks?": 2,
    "What was the return on equity?": 5,
    "What is the liquidity coverage ratio?": 7,
}

for k in [1, 2, 3, 5]:
    hits = 0
    for q, gt_idx in ground_truth.items():
        results = store.retrieve(q, k=k)
        retrieved_ids = [r["doc_id"] for r in results]
        if gt_idx in retrieved_ids:
            hits += 1
    recall = hits / len(ground_truth)
    print(f"Recall@{k}: {recall:.2f}  ({hits}/{len(ground_truth)} queries)")

Recall increases with $k$ but with diminishing returns. Sending more chunks to the LLM improves coverage but adds tokens and introduces noise — irrelevant context can degrade answer quality as badly as missing context. The optimal $k$ balances recall against context window cost; typically $k = 3$ to $k = 5$ in production.

## Putting It Together — Minimal RAG Pipeline

We assemble the full RAG pipeline: retrieve context for a query, format it with inline citation markers, and generate a grounded answer. This is the "hello world" of RAG — in [notebook 07](/courses/llm-eng/07-rag-pipeline.html) we replace the numpy store with ChromaDB and the inline corpus with real SEC filings.

The system prompt is the contract between the retrieval system and the model: it specifies that the model must answer from context only and cite sources. Without this contract, the model freely mixes retrieved facts with parametric knowledge, making outputs unauditable.

We define a `rag_query` function that handles the full retrieve-format-generate loop:

In [ ]:
def rag_query(
    question: str,
    store: VectorStore,
    llm: LLMClient,
    k: int = 3,
) -> str:
    """Retrieve relevant chunks and generate a grounded answer with citations."""
    results = store.retrieve(question, k=k)

    context_blocks = []
    for i, r in enumerate(results):
        context_blocks.append(f"[{i+1}] {r['text']}")   # <1>
    context = "\n\n".join(context_blocks)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a financial analyst assistant. "
                "Answer the user's question using ONLY the provided context. "
                "Cite sources using [N] notation. "
                "If the context does not contain enough information, say so explicitly."
            ),
        },
        {
            "role": "user",
            "content": f"Context:\n{context}\n\nQuestion: {question}",
        },
    ]
    return llm.complete(messages)

1. We assign each retrieved chunk a citation number `[1]`, `[2]`, `[3]` so the model can reference them inline — making the answer auditable and traceable to source passages.

Running the pipeline over three test questions:

In [ ]:
test_questions = [
    "What was the CET1 capital ratio at year-end?",
    "How did investment banking perform compared to the prior year?",
    "What is the company's return on equity and book value per share?",
]
for q in test_questions:
    print(f"Q: {q}")
    print(f"A: {rag_query(q, store, llm)}\n")

print(f"Total API cost: ${llm.total_cost:.5f}")

:::{.callout-important}
The system prompt instruction "Answer using ONLY the provided context" is essential for faithfulness. Without it, the model will supplement retrieved facts with its pretrained knowledge — making it impossible to distinguish what came from your documents from what the model invented.

:::

## Appendix: BM25 and Hybrid Retrieval

Dense embeddings capture semantic similarity but miss exact-match lookups. Ticker symbols like "JPM", regulation codes like "Reg T", or CUSIP numbers have no semantic meaning — they are arbitrary identifiers, and a dense model may confuse them with similar-looking strings. **BM25** (Best Match 25) is a classic sparse retrieval algorithm based on term frequency and inverse document frequency:

$$\text{BM25}(q, d) = \sum_{t \in q} \text{IDF}(t) \cdot \frac{f(t,d) \cdot (k_1 + 1)}{f(t,d) + k_1 \cdot (1 - b + b \cdot |d|/\text{avgdl})}$$

where $f(t, d)$ is the term frequency of token $t$ in document $d$, $|d|$ is the document length, $\text{avgdl}$ is the average document length over the corpus, $k_1 \approx 1.5$ controls term frequency saturation, and $b \approx 0.75$ controls length normalization. BM25 scores high when the query term appears frequently in a short document relative to the corpus average.

**Hybrid retrieval** combines dense and sparse scores using **Reciprocal Rank Fusion** (RRF), which is robust to differences in score scale between systems:

$$\text{RRF}(d) = \sum_i \frac{1}{k + \text{rank}_i(d)}$$

where $k = 60$ is a smoothing constant and $\text{rank}_i(d)$ is the rank of document $d$ in retrieval system $i$. RRF requires only rankings, not raw scores, so it needs no calibration. In practice, hybrid retrieval outperforms either method alone on financial corpora where exact identifier matching and semantic understanding are both required. We implement hybrid retrieval in [notebook 07](/courses/llm-eng/07-rag-pipeline.html) using the `rank_bm25` library.

---

$\blacksquare$